In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import linear_model

In [2]:
#"KGAT_d9332ed125004b542a3ae9b21c80d300"
#API key that I might need form kaggle
#https://www.kaggle.com/datasets/anuragupadhyaya/anticancer-peptides-data-set?select=ACPs_Lung_cancer.csv

In [6]:
csv_lung_cancer_url = "https://www.kaggle.com/datasets/anuragupadhyaya/anticancer-peptides-data-set?select=ACPs_Lung_cancer.csv"
csv_breast_cancer_url = "https://www.kaggle.com/datasets/anuragupadhyaya/anticancer-peptides-data-set?select=ACPs_Breast_cancer.csv"

df = pd.read_csv(csv_lung_cancer_url)

ParserError: Error tokenizing data. C error: Expected 1 fields in line 9, saw 2


In [5]:
#df = pd.read_csv('ACPs_Lung_cancer.csv')
#df2 = pd.read_csv('ACPs_Breast_cancer.csv')

In [ ]:
df.head()
#df2.head()
df.info()
df['class'].value_counts()

In [ ]:
df['seq_len'] = df['sequence'].apply(len)
plt.figure(figsize=(6, 4))
sns.histplot(df['seq_len'], bins=30)
plt.title("Length Distribution of Peptides")
plt.show()

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(x='class', data = df)
plt.title("Class/Activity Distribution")
plt.show()

In [ ]:
def encode_sequence(seq, max_len = 50):
  amino_acid_encode = {
    'A': 1,
    'C': 2,
    'D': 3,
    'E': 4,
    'F': 5,
    'G': 6,
    'H': 7,
    'I': 8,
    'K': 9,
    'L': 10,
    'M': 11,
    'N': 12,
    'P': 13,
    'Q': 14,
    'R': 15,
    'S': 16,
    'T': 17,
    'V': 18,
    'W': 19,
    'Y': 20
  }
  encoded = []
  for letter in seq:
    encoded.append(amino_acid_encode[letter])

  #padding with int 0
  if len(encoded) < max_len:
    for i in range(max_len-len(encoded)):
      encoded.append(0)
  return encoded


In [ ]:
X = np.array(df['sequence'].apply(lambda s : encode_sequence(s)).tolist(), dtype = np.int64)

In [ ]:
def encode_activity(activity):
  label_encode = {
    "inactive - exp": 0,
    "inactive - virtual" : 0,
    "mod. active": 1,
    "very active": 2
  }
  encoded_activity = []
  encoded_activity.append(label_encode[activity])
  return encoded_activity

In [ ]:
y = np.array(df['class'].apply(lambda s : encode_activity(s)).tolist(), dtype = np.int64)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify = y,
    random_state=42
)

In [ ]:
import torch

X_train_t = torch.tensor(X_train, dtype = torch.long)
y_train_t = torch.tensor(y_train, dtype = torch.long)

X_test_t = torch.tensor(X_test, dtype = torch.long)
y_test_t = torch.tensor(y_test, dtype = torch.long)


In [ ]:
from torch.utils.data import Dataset, DataLoader
class PeptideDataset(Dataset):
  def __init__(self, X, y):
    self.X = X
    self.y = y
  def __len__(self):
    return len(self.y)
  def __getitem__(self, index):
    return self.X[index], self.y[index]




In [ ]:
train_ds = PeptideDataset(X_train_t, y_train_t)
test_ds = PeptideDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32)

In [64]:
import torch.nn as nn
import torch.nn.functional as F

class PeptideClassifier(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedding = nn.Embedding(
        num_embeddings = 21,
        embedding_dim=32,
        padding_idx = 0
    )
    self.fc1 = nn.Linear(32, 64)
    self.fc2 = nn.Linear(64, 3)

  def forward(self, x):
    x = self.embedding(x)
    x = x.mean(dim=1)
    x = F.relu(self.fc1(x))
    return self.fc2(x)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = PeptideClassifier().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [ ]:
def train_epoch(model, loader):
  model.train()
  total_loss = 0

  for X, y in loader:
    X, y = X.to(device), y.to(device)

    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

  return total_loss / len(loader)


def evaluate(model, loader):
  model.eval()
  correct, total = 0, 0
  with torch.no_grad():
    for X, y in loader:
      X, y = X.to(device), y.to(device)
      preds = model(X).argmax(dim = 1)
      correct += (preds == y).sum().item()
      total += y.size(0)

  return correct / total

In [ ]:
for epoch in range(10):
  loss = train_epoch(model, train_loader)
  acc = evaluate(model, test_loader)
  print(f"Epoch {epoch+1}: loss= {loss:.4f}, acc = {acc:.3f}")